# `ExcelDataSource` - Lecture et écriture de fichiers Excel

**Module :** `kadi.kidas.sources.excel_source`  
**Classe :** `ExcelDataSource`

---

## Introduction

`ExcelDataSource` gère les fichiers Excel (`.xls`, `.xlsx`) tels qu'on les trouve dans les rapports agricoles béninois :

- **Cellules fusionnées** : colonnes Commune/Marché fusionnées sur plusieurs lignes → résolues par `forward fill`
- **En-têtes non standard** : en-tête à la 3ème ou 5ème ligne → détection automatique
- **Multi-feuilles** : chaque mois/région dans une feuille séparée
- **Lignes vides** : lignes de séparation automatiquement supprimées

## 1. Création des fichiers de démonstration

In [1]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('../../'))

import os
import tempfile
import pandas as pd

from kadi.kidas.sources.excel_source import ExcelDataSource

# --- Fichier Excel simple (une feuille) ---
df_simple = pd.DataFrame({
    'culture': ['maïs', 'niébé', 'igname', 'sorgho'],
    'commune': ['Cotonou', 'Parakou', 'Bohicon', 'Kandi'],
    'rendement_kg': [1500, 800, 2000, 1200],
    'date': ['2024-01', '2024-01', '2024-02', '2024-02'],
})

fichier_simple = os.path.join(tempfile.gettempdir(), 'recoltes_simple.xlsx')
df_simple.to_excel(fichier_simple, index=False, sheet_name='Recoltes')

# --- Fichier Excel multi-feuilles ---
fichier_multi = os.path.join(tempfile.gettempdir(), 'recoltes_multi.xlsx')
with pd.ExcelWriter(fichier_multi, engine='openpyxl') as writer:
    df_simple.iloc[:2].to_excel(writer, sheet_name='Janvier', index=False)
    df_simple.iloc[2:].to_excel(writer, sheet_name='Fevrier', index=False)

print(f"Fichier simple    : {fichier_simple}")
print(f"Fichier multi     : {fichier_multi}")

Fichier simple    : /tmp/recoltes_simple.xlsx
Fichier multi     : /tmp/recoltes_multi.xlsx


#### Ce que produit cette cellule

```
Fichier simple    : /tmp/recoltes_simple.xlsx
Fichier multi     : /tmp/recoltes_multi.xlsx
```

Deux fichiers Excel sont créés dans le dossier temporaire du système (`/tmp/`) :

**Fichier simple** (`recoltes_simple.xlsx`) — une seule feuille nommée `Recoltes` contenant 4 lignes de données agricoles :

| culture | commune | rendement_kg | date |
|---|---|---|---|
| maïs | Cotonou | 1500 | 2024-01 |
| niébé | Parakou | 800 | 2024-01 |
| igname | Bohicon | 2000 | 2024-02 |
| sorgho | Kandi | 1200 | 2024-02 |

**Fichier multi-feuilles** (`recoltes_multi.xlsx`) — deux feuilles issues du même DataFrame, découpé par mois :
- Feuille `Janvier` : les 2 premières lignes (maïs, niébé)
- Feuille `Fevrier` : les 2 dernières lignes (igname, sorgho)

Ce découpage par mois imite fidèlement les classeurs Excel produits par les services agricoles béninois, où chaque mois ou chaque département occupe une feuille distincte.

## 2. Initialisation de `ExcelDataSource`

```python
ExcelDataSource(
    file_path: str,
    sheet_name: str | int = 0,
    header_row: str | int = 'auto',
)
```

In [2]:
# Initialisation avec auto-détection de l'en-tête
source = ExcelDataSource(fichier_simple)
print(repr(source))

ExcelDataSource(source_path='/tmp/recoltes_simple.xlsx', source_type='excel', encoding='utf-8')


#### Ce que produit cette cellule

```
ExcelDataSource(source_path='/tmp/recoltes_simple.xlsx', source_type='excel', encoding='utf-8')
```

La représentation de l'objet confirme les trois informations clés de l'instance au moment de sa création :

| Champ | Valeur | Ce que ça signifie |
|---|---|---|
| `source_path` | `/tmp/recoltes_simple.xlsx` | Le chemin complet vers le fichier Excel. |
| `source_type` | `excel` | Le type de source — toujours `excel` pour cette classe. |
| `encoding` | `utf-8` | Contrairement aux CSV, Excel stocke le texte en Unicode nativement. L'encodage est donc connu d'emblée sans analyse : c'est toujours UTF-8. |

Notez que la feuille active et la ligne d'en-tête ne sont pas encore résolues ici : elles le seront lors du premier appel à `read()`.

In [3]:
# Initialisation avec paramètres explicites
source_explicite = ExcelDataSource(
    file_path=fichier_simple,
    sheet_name='Recoltes',
    header_row=0,
)
print(repr(source_explicite))

ExcelDataSource(source_path='/tmp/recoltes_simple.xlsx', source_type='excel', encoding='utf-8')


#### Ce que produit cette cellule

```
ExcelDataSource(source_path='/tmp/recoltes_simple.xlsx', source_type='excel', encoding='utf-8')
```

La représentation est **identique** à celle de la cellule précédente, bien que les paramètres soient ici explicites (`sheet_name='Recoltes'`, `header_row=0`).

Pourquoi le même résultat ? Le `repr` de la classe expose uniquement les champs persistants communs (`source_path`, `source_type`, `encoding`). Les paramètres `sheet_name` et `header_row` sont des options de lecture stockées en interne mais non affichées dans la représentation.

| Paramètre | Valeur passée | Effet |
|---|---|---|
| `sheet_name='Recoltes'` | Nom explicite | Lira la feuille `Recoltes` sans avoir à la détecter. |
| `header_row=0` | Ligne 0 | L'en-tête est à la première ligne — aucune détection automatique nécessaire. |

Le mode explicite est utile quand vous connaissez à l'avance la structure du fichier. Le mode auto (cellule précédente) est préférable pour des fichiers dont la structure varie.

## 3. `validate_connection()` - Vérifier l'accessibilité

In [4]:
source = ExcelDataSource(fichier_simple)
print(f"Fichier accessible       : {source.validate_connection()}")

source_absent = ExcelDataSource('/absent.xlsx')
print(f"Fichier absent accessible : {source_absent.validate_connection()}")

Le fichier Excel '/absent.xlsx' n'existe pas ou n'est pas lisible.


Fichier accessible       : True
Fichier absent accessible : False


#### Ce que produit cette cellule

```
[erreur standard] Le fichier Excel '/absent.xlsx' n'existe pas ou n'est pas lisible.
[sortie standard] Fichier accessible       : True
[sortie standard] Fichier absent accessible : False
```

Trois lignes apparaissent, mais elles proviennent de deux canaux différents :

| Canal | Message | Origine |
|---|---|---|
| Erreur standard (rouge dans Jupyter) | `... n'existe pas ou n'est pas lisible.` | Le logger interne de `ExcelDataSource` signale le problème en avertissement technique avant que le booléen ne soit retourné. |
| Sortie standard | `Fichier accessible : True` | `/tmp/recoltes_simple.xlsx` existe bien sur le disque. |
| Sortie standard | `Fichier absent accessible : False` | `/absent.xlsx` n'existe pas — la méthode retourne `False` sans lever d'exception. |

**Comportement clé :** `validate_connection()` ne plante jamais. Elle retourne un booléen et laisse le programme décider de la suite. C'est vous qui choisissez d'arrêter ou de continuer selon le résultat.

L'avertissement apparaît avant les lignes de résultat car Python traite la sortie standard et l'erreur standard comme deux tuyaux indépendants, et Jupyter les affiche dans l'ordre de réception des octets.

## 4. `list_sheets()` - Lister les feuilles disponibles

In [5]:
# Feuilles du fichier simple
source_simple = ExcelDataSource(fichier_simple)
print(f"Feuilles (simple) : {source_simple.list_sheets()}")

# Feuilles du fichier multi-feuilles
source_multi = ExcelDataSource(fichier_multi)
print(f"Feuilles (multi)  : {source_multi.list_sheets()}")

Feuilles (simple) : ['Recoltes']
Feuilles (multi)  : ['Janvier', 'Fevrier']


#### Ce que produit cette cellule

```
Feuilles (simple) : ['Recoltes']
Feuilles (multi)  : ['Janvier', 'Fevrier']
```

`list_sheets()` ouvre le classeur Excel et retourne la liste ordonnée de toutes ses feuilles, dans l'ordre où elles apparaissent dans le fichier.

| Fichier | Feuilles retournées | Explication |
|---|---|---|
| `recoltes_simple.xlsx` | `['Recoltes']` | Une seule feuille, créée avec `sheet_name='Recoltes'` en cellule 1. |
| `recoltes_multi.xlsx` | `['Janvier', 'Fevrier']` | Deux feuilles dans l'ordre de création : d'abord janvier, puis février. |

Cette méthode est particulièrement utile pour explorer un classeur inconnu avant de décider quelle feuille lire. Dans les rapports agricoles mensuels, on la parcourt souvent en boucle pour traiter toutes les feuilles d'un coup.

## 5. `detect_header_row()` - Détection automatique de la ligne d'en-tête

In [6]:
source = ExcelDataSource(fichier_simple, header_row='auto')
ligne_entete = source.detect_header_row()
print(f"Ligne d'en-tête détectée (0-indexé) : {ligne_entete}")

Ligne d'en-tête détectée (0-indexé) : 0


#### Ce que produit cette cellule

```
Ligne d'en-tête détectée (0-indexé) : 0
```

`detect_header_row()` analyse le fichier Excel pour trouver la ligne qui contient les noms de colonnes.

**Résultat ici :** La valeur `0` signifie que l'en-tête se trouve à la **première ligne** du fichier (ligne index 0 en Python), ce qui correspond à la ligne Excel numéro 1.

**Comment fonctionne cette détection ?**

La méthode calcule un score de densité pour chaque ligne candidate. Une ligne d'en-tête typique contient principalement du texte (peu ou pas de valeurs numériques), tandis que les lignes de données mélangent texte et chiffres. La ligne avec le meilleur score textuel est sélectionnée comme en-tête.

**Pourquoi cette fonctionnalité est importante ?**

Beaucoup de rapports Excel béninois insèrent un titre ou un logo en haut du fichier avant le tableau de données. Sans détection automatique, lire naïvement depuis la ligne 0 produirait un DataFrame avec des noms de colonnes incorrects. `detect_header_row()` gère ces cas sans aucun paramétrage manuel.

## 6. `unmerge_cells()` - Résolution des cellules fusionnées

Les fichiers Excel africains ont souvent la colonne *Commune* ou *Marché* avec des cellules fusionnées :

| Commune   | Marché    | Culture |
|-----------|-----------|--------|
| Cotonou   | Dantokpa  | Maïs   |
| *(vide)*  | Tokpa     | Niébé  |
| Parakou   | Central   | Sorgho |

In [7]:
import numpy as np

# Simulation d'un DataFrame avec cellules fusionnées (NaN = fusion)
df_fusionne = pd.DataFrame({
    'commune': ['Cotonou', None, None, 'Parakou', None],
    'marche':  ['Dantokpa', 'Tokpa', 'Missebo', 'Central', 'Malanville'],
    'culture': ['Maïs', 'Niébé', 'Igname', 'Sorgho', 'Manioc'],
})

print("AVANT forward fill (cellules fusionnées simulées):")
print(df_fusionne.to_string())

# Application de unmerge_cells()
source = ExcelDataSource(fichier_simple)
df_resolu = source.unmerge_cells(df_fusionne)

print("\nAPRES forward fill :")
print(df_resolu.to_string())

AVANT forward fill (cellules fusionnées simulées):
   commune      marche culture
0  Cotonou    Dantokpa    Maïs
1      NaN       Tokpa   Niébé
2      NaN     Missebo  Igname
3  Parakou     Central  Sorgho
4      NaN  Malanville  Manioc

APRES forward fill :
   commune      marche culture
0  Cotonou    Dantokpa    Maïs
1  Cotonou       Tokpa   Niébé
2  Cotonou     Missebo  Igname
3  Parakou     Central  Sorgho
4  Parakou  Malanville  Manioc


#### Ce que produit cette cellule

```
AVANT forward fill (cellules fusionnées simulées):
   commune      marche culture
0  Cotonou    Dantokpa    Maïs
1      NaN       Tokpa   Niébé
2      NaN     Missebo  Igname
3  Parakou     Central  Sorgho
4      NaN  Malanville  Manioc

APRES forward fill :
   commune      marche culture
0  Cotonou    Dantokpa    Maïs
1  Cotonou       Tokpa   Niébé
2  Cotonou     Missebo  Igname
3  Parakou     Central  Sorgho
4  Parakou  Malanville  Manioc
```

Cette démonstration illustre le problème des cellules fusionnées et sa résolution.

**Avant le traitement :** Quand pandas lit un fichier Excel avec des cellules fusionnées, seule la première cellule de la fusion contient la valeur — les autres deviennent des `NaN`. C'est ce qu'on voit aux lignes 1, 2 et 4 dans la colonne `commune`.

**Après le traitement :** `unmerge_cells()` applique un *forward fill* (propagation vers le bas) sur les colonnes concernées. Chaque `NaN` est remplacé par la dernière valeur non nulle rencontrée au-dessus.

| Ligne | Avant | Après | Explication |
|---|---|---|---|
| 1 | `NaN` | `Cotonou` | La valeur est propagée depuis la ligne 0. |
| 2 | `NaN` | `Cotonou` | Idem, toujours dans le bloc Cotonou. |
| 4 | `NaN` | `Parakou` | Propagée depuis la ligne 3, nouveau bloc. |

La colonne `marche` et la colonne `culture` restent intactes, car elles ne contiennent pas de `NaN`.

Cette correction est indispensable pour pouvoir filtrer ou regrouper les données par commune sans se retrouver avec des valeurs manquantes là où Excel affichait une cellule fusionnée.

## 7. `read()` - Lecture d'une feuille Excel

In [8]:
# Lecture de la première feuille (par défaut)
source = ExcelDataSource(fichier_simple)
df = source.read()
print(f"Feuille lue : {len(df)} lignes × {len(df.columns)} colonnes")
df

Feuille lue : 4 lignes × 4 colonnes


,culture,commune,rendement_kg,date
0,maïs,Cotonou,1500,2024-01
1,niébé,Parakou,800,2024-01
2,igname,Bohicon,2000,2024-02
3,sorgho,Kandi,1200,2024-02


#### Ce que produit cette cellule

```
Feuille lue : 4 lignes × 4 colonnes
```

Suivi d'un tableau affiché avec 4 lignes et 4 colonnes.

**Analyse du tableau :**

| Colonne | Type inféré | Observation |
|---|---|---|
| `culture` | texte | Noms de cultures en français avec accents — lus correctement car Excel stocke l'Unicode nativement. |
| `commune` | texte | Noms des communes béninoises : Cotonou, Parakou, Bohicon, Kandi. |
| `rendement_kg` | entier | Valeurs entières (1500, 800, 2000, 1200) — inférées comme `int64`. |
| `date` | texte | Les dates au format `YYYY-MM` sont lues comme des chaînes. Pour les convertir en dates réelles, il faudrait utiliser `pd.to_datetime()` après lecture. |

**Ce que fait `read()` en interne :**
1. Appel de `detect_header_row()` pour trouver la ligne d'en-tête.
2. Lecture via `pd.read_excel()` avec les bons paramètres.
3. Suppression des lignes entièrement vides.
4. Application de `unmerge_cells()` si des colonnes ont des `NaN` consécutifs.
5. Mise à jour de `source.last_read` avec l'heure courante.

Le résultat correspond exactement aux données créées en cellule 1 — la lecture est fidèle et sans perte.

In [9]:
# Lecture d'une feuille spécifique par son nom
source_multi = ExcelDataSource(fichier_multi)
df_janvier = source_multi.read(sheet_name='Janvier')
print("Feuille 'Janvier' :")
print(df_janvier.to_string())

df_fevrier = source_multi.read(sheet_name='Fevrier')
print("\nFeuille 'Fevrier' :")
print(df_fevrier.to_string())

Feuille 'Janvier' :
  culture  commune  rendement_kg     date
0    maïs  Cotonou          1500  2024-01
1   niébé  Parakou           800  2024-01

Feuille 'Fevrier' :
  culture  commune  rendement_kg     date
0  igname  Bohicon          2000  2024-02
1  sorgho    Kandi          1200  2024-02


#### Ce que produit cette cellule

```
Feuille 'Janvier' :
  culture  commune  rendement_kg     date
0    maïs  Cotonou          1500  2024-01
1   niébé  Parakou           800  2024-01

Feuille 'Fevrier' :
  culture  commune  rendement_kg     date
0  igname  Bohicon          2000  2024-02
1  sorgho    Kandi          1200  2024-02
```

La méthode `read(sheet_name=...)` permet de cibler une feuille précise par son nom dans un classeur multi-feuilles.

**Résultat par feuille :**

| Feuille | Lignes | Cultures | Communes | Période |
|---|---|---|---|---|
| `Janvier` | 2 | maïs, niébé | Cotonou, Parakou | 2024-01 |
| `Fevrier` | 2 | igname, sorgho | Bohicon, Kandi | 2024-02 |

Chaque feuille retourne un DataFrame indépendant avec son propre index commençant à 0. Les données correspondent exactement au découpage fait lors de la création en cellule 1 (`df_simple.iloc[:2]` pour janvier, `df_simple.iloc[2:]` pour février).

Pour traiter toutes les feuilles d'un classeur mensuel d'un coup, on peut les lire en boucle en utilisant `list_sheets()` puis `read(sheet_name=...)` pour chacune, avant de les concaténer avec `pd.concat()`.

## 8. `get_sheet_metadata()` - Métadonnées d'une feuille

In [10]:
source_multi = ExcelDataSource(fichier_multi)
meta_jan = source_multi.get_sheet_metadata('Janvier')
print("Métadonnées de la feuille 'Janvier' :")
for cle, valeur in meta_jan.items():
    print(f"  {cle:12s} : {valeur}")

Métadonnées de la feuille 'Janvier' :
  sheet_name   : Janvier
  rows         : 2
  cols         : 4
  columns      : ['culture', 'commune', 'rendement_kg', 'date']


#### Ce que produit cette cellule

```
Métadonnées de la feuille 'Janvier' :
  sheet_name   : Janvier
  rows         : 2
  cols         : 4
  columns      : ['culture', 'commune', 'rendement_kg', 'date']
```

`get_sheet_metadata()` retourne un résumé structuré d'une feuille précise, **sans charger toutes les données en mémoire**.

| Champ | Valeur | Ce que ça signifie |
|---|---|---|
| `sheet_name` | `Janvier` | Le nom de la feuille analysée. |
| `rows` | `2` | La feuille Janvier contient 2 lignes de données (sans compter l'en-tête). |
| `cols` | `4` | Elle a 4 colonnes. |
| `columns` | `['culture', 'commune', 'rendement_kg', 'date']` | La liste des noms de colonnes, dans leur ordre d'apparition dans le fichier. |

Cette méthode est idéale pour **explorer la structure** d'un classeur avant de le lire entièrement, ou pour **vérifier** que les colonnes attendues sont bien présentes avant de lancer un pipeline de traitement.

## 9. `write()` - Écriture vers Excel

In [11]:
df_a_ecrire = pd.DataFrame({
    'culture': ['fonio', 'soja'],
    'prix_xof': [420, 600],
})

fichier_sortie = os.path.join(tempfile.gettempdir(), 'sortie_excel.xlsx')
source_sortie = ExcelDataSource(fichier_sortie)
succes = source_sortie.write(df_a_ecrire, sheet_name='Cultures')
print(f"Écriture réussie : {succes}")

# Vérification
df_relu = source_sortie.read()
df_relu

Écriture réussie : True


,culture,prix_xof
0,fonio,420
1,soja,600


#### Ce que produit cette cellule

```
Écriture réussie : True
```

Suivi du tableau relu depuis le fichier Excel qui vient d'être écrit.

**Déroulement en deux temps :**

**Écriture** — `write()` crée le fichier `sortie_excel.xlsx` avec une feuille nommée `Cultures` et y insère le DataFrame suivant :

| culture | prix_xof |
|---|---|
| fonio | 420 |
| soja | 600 |

La valeur `True` confirme que l'opération s'est terminée sans erreur.

**Relecture** — `read()` ouvre immédiatement le fichier qui vient d'être écrit et retourne les mêmes données. Cela confirme que la sérialisation (écriture) et la désérialisation (lecture) sont cohérentes.

**Points à noter :**
- `write()` utilise le moteur `openpyxl` pour produire des fichiers `.xlsx` compatibles avec Excel, LibreOffice et Google Sheets.
- Si le fichier cible existe déjà, il est **remplacé entièrement**.
- L'index pandas n'est pas écrit dans le fichier (comportement `index=False` par défaut), ce qui explique l'absence d'une colonne numérique parasite dans le résultat.

## 10. `get_metadata()` - Métadonnées du fichier

In [12]:
source = ExcelDataSource(fichier_multi)
source.read()
meta = source.get_metadata()
for cle, valeur in meta.items():
    print(f"  {cle:25s} : {valeur}")

  source_path               : /tmp/recoltes_multi.xlsx
  source_type               : excel
  sheets                    : ['Janvier', 'Fevrier']
  active_sheet              : 0
  detected_header_row       : 0
  size_kb                   : 5.4
  last_read                 : 2026-07-07T18:17:37.283617


#### Ce que produit cette cellule

```
  source_path               : /tmp/recoltes_multi.xlsx
  source_type               : excel
  sheets                    : ['Janvier', 'Fevrier']
  active_sheet              : 0
  detected_header_row       : 0
  size_kb                   : 5.4
  last_read                 : 2026-07-07T18:02:42.728714
```

`get_metadata()` retourne un dictionnaire de 7 champs qui décrivent complètement le fichier et l'état de la source.

**Détail de chaque champ :**

| Champ | Valeur | Ce que ça signifie |
|---|---|---|
| `source_path` | `/tmp/recoltes_multi.xlsx` | Chemin complet vers le fichier analysé. |
| `source_type` | `excel` | Type de source — toujours `excel` pour cette classe. |
| `sheets` | `['Janvier', 'Fevrier']` | Liste de toutes les feuilles du classeur, dans leur ordre d'apparition. |
| `active_sheet` | `0` | Index de la feuille utilisée par défaut lors d'un `read()` sans paramètre. `0` correspond à la première feuille, ici `Janvier`. |
| `detected_header_row` | `0` | La ligne d'en-tête a été détectée à l'index 0 (première ligne), ce qui est la position habituelle pour un fichier bien formé. |
| `size_kb` | `5.4` | Taille du fichier Excel en kilo-octets. Plus lourd qu'un CSV équivalent (0.23 Ko) car le format `.xlsx` est une archive ZIP contenant du XML. |
| `last_read` | `2026-07-07T18:02:42...` | Horodatage de la dernière lecture, mis à jour par `source.read()` juste avant cet appel. |

Ces métadonnées font partie du rapport de traçabilité généré automatiquement par `DataPipeline` à chaque exécution de pipeline.

## Récapitulatif des méthodes

| Méthode | Description |
|---|---|
| `validate_connection()` | Vérifie l'existence du fichier |
| `list_sheets()` | Liste toutes les feuilles du classeur |
| `detect_header_row()` | Détecte la ligne d'en-tête par score de densité |
| `unmerge_cells(df)` | Résout les cellules fusionnées par forward fill |
| `read(sheet_name)` | Lit une feuille avec en-tête auto et déduplication NaN |
| `get_sheet_metadata(sheet)` | Retourne lignes/colonnes/colonnes d'une feuille |
| `write(data, sheet_name)` | Écrit un DataFrame dans le fichier Excel |
| `get_metadata()` | Retourne feuilles, taille, feuille active |